In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



In [3]:
#import MAP716 data
MAP716=pd.read_csv('MAP716_CAT2.csv')
MAP716.head(2)


,District,facilityname,PID,year,Gender,ANCServices,DeliveryServices,PostNatalCare,WASHServices,last_status,cd4_baseline_count,cd4_baseline_date
0,B,Kenyatta Hosp,2407,2009,Female,No,Yes,No,Yes,Active,9,3/28/2013 0:00
1,B,Mbagathi Dist Hosp,1500,2008,Female,Yes,No,Yes,No,Active,12,8/14/2013 0:00


In [4]:
#Identify and describe the data types and variables included in the dataset.
MAP716.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5255 entries, 0 to 5254
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   District            5255 non-null   object
 1   facilityname        5255 non-null   object
 2   PID                 5255 non-null   int64 
 3   year                5255 non-null   int64 
 4   Gender              5255 non-null   object
 5   ANCServices         5255 non-null   object
 6   DeliveryServices    5255 non-null   object
 7   PostNatalCare       5255 non-null   object
 8   WASHServices        5255 non-null   object
 9   last_status         5255 non-null   object
 10  cd4_baseline_count  5255 non-null   int64 
 11  cd4_baseline_date   1365 non-null   object
dtypes: int64(3), object(9)
memory usage: 492.8+ KB


In [17]:
# specify outcome variable 'last-status' and independent variables
outcome=MAP716['last_status']
outcome.head(1)

independent_variables=MAP716[['District','Gender','ANCServices','DeliveryServices','PostNatalCare','WASHServices','cd4_baseline_count']]

In [6]:
#Data Preparation|check missing values
MAP716.isnull().sum()

,0
District,0
facilityname,0
PID,0
year,0
Gender,0
ANCServices,0
DeliveryServices,0
PostNatalCare,0
WASHServices,0
last_status,0


In [7]:
#mode of cd4baseline date
cd4baseline_date=MAP716['cd4_baseline_date'].mode()
cd4baseline_date

,cd4_baseline_date
0,9/15/2011 0:00


In [8]:
#Data Preparation|fill the missing rows in cd4 baseline date with 15th June 2011
MAP716_New=MAP716['cd4_baseline_date'].fillna('15/06/2011',inplace=True)




/tmp/ipython-input-2674693764.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  MAP716_New=MAP716['cd4_baseline_date'].fillna('15/06/2011',inplace=True)


In [9]:
#Data validation|recheck missing values
MAP716.isnull().sum()

,0
District,0
facilityname,0
PID,0
year,0
Gender,0
ANCServices,0
DeliveryServices,0
PostNatalCare,0
WASHServices,0
last_status,0


In [10]:
#Data Preparation|check duplicates in data
MAP716.duplicated().sum()


np.int64(0)

In [11]:
#Data Analysis, Descriptive statistics|describe the cd4 baseline count
Descriptive=MAP716['cd4_baseline_count'].describe()
Descriptive
#save decriptive to excel
Descriptive.to_excel('CD4BaselineCounts.xlsx')

In [12]:
#Data Analysis, Descriptive statistics(frequency analysis) for categorical variables
#create columns for the data
MAP716_col=MAP716[['Gender','District', 'ANCServices','DeliveryServices','PostNatalCare','WASHServices']]

In [13]:
# Define column optons to generate univariate analysis
MAP716_Univar = ['No', 'Yes']

# Initialize a list to hold data
summary_list = []

# Loop through each indicator
for col in MAP716_col.columns[2:]:
    counts = MAP716_col[col].value_counts(normalize=True) * 100
    counts = counts.reindex(MAP716_Univar, fill_value=0)

    summary_list.append({
        'Indicator': col,
        'No (%)': counts['No'],
        'Yes (%)': counts['Yes']
    })

# Convert to DataFrame
univariate_likert_df = pd.DataFrame(summary_list)

# Round percentages
univariate_likert_df[['No (%)','Yes (%)']] = univariate_likert_df[['No (%)','Yes (%)']].round(1)

# Display
print(univariate_likert_df)

# --- Save to Excel ---
univariate_likert_df.to_excel("MAP716_univariate_summary.xlsx")

          Indicator  No (%)  Yes (%)
0       ANCServices    19.0     81.0
1  DeliveryServices    49.9     50.1
2     PostNatalCare    36.5     63.5
3      WASHServices    84.7     15.3


In [14]:
# Define options order for Bivariate cross tabs
MAP716_order = ['No', 'Yes']

# Initialize a list to hold data
summary_list = []

# Loop through each District and each indicator
for District, group in MAP716_col.groupby('District'):
    for col in MAP716_col.columns[2:]:  # skip 'District'
        counts = group[col].value_counts(normalize=True) * 100
        counts = counts.reindex(MAP716_order, fill_value=0)

        summary_list.append({
            'Indicator': col,
            'District': District,
            'No (%)': counts['No'],
            'Yes (%)': counts['Yes']
        })

# Convert to DataFrame
bivariate_crosstabs= pd.DataFrame(summary_list)

# Round percentages
bivariate_crosstabs[['No (%)','Yes (%)']] = bivariate_crosstabs[['No (%)','Yes (%)']].round(1)

# -- Pivot so indicators as rows ---
bivariate_pivot = bivariate_crosstabs.pivot(index='Indicator', columns='District', values=['No (%)', 'Yes (%)'])

# --- Optional: reorder column levels so top = "Response" ("Yes"/"No") ---
bivariate_pivot = bivariate_pivot.swaplevel(axis=1)
bivariate_pivot = bivariate_pivot.sort_index(axis=1, level=0)  # ensures "No"/"Yes" grouped

# Display
print(bivariate_pivot)

# --- Save to Excel ---
bivariate_pivot.to_excel("MAP716_Bivariate_District.xlsx")




District              A              B        
                 No (%) Yes (%) No (%) Yes (%)
Indicator                                     
ANCServices        16.9    83.1   20.8    79.2
DeliveryServices   43.8    56.2   55.2    44.8
PostNatalCare      51.8    48.2   23.3    76.7
WASHServices       83.1    16.9   86.1    13.9


In [15]:
#BIVARIATE ANALYSIS perform chi-square test on variables ANCServices and WASHServices
from scipy.stats import chi2_contingency

# Create a contingency table from the categorical columns
contingency_table = pd.crosstab(MAP716['ANCServices'], MAP716['WASHServices'])

# Perform the chi-square test on the contingency table
chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2}")
print(f"P-value: {p}")
print(f"Degrees of freedom: {dof}")
print("Expected frequencies:")
print(expected)
#save to excel
contingency_table.to_excel('ANC_WASH.xlsx')




Chi-square statistic: 4042.753157363823
P-value: 0.0
Degrees of freedom: 1
Expected frequencies:
[[ 845.30884872  152.69115128]
 [3605.69115128  651.30884872]]


In [16]:

#(MULTIVARIATE ANALYSIS)perform binary logistic regression with last status as dependent variable
import statsmodels.api as sm

# Convert categorical independent variables to numerical using one-hot encoding
independent_variables_encoded = pd.get_dummies(independent_variables, drop_first=True)

# Convert boolean columns to integer (0 or 1) explicitly
for col in independent_variables_encoded.select_dtypes(include='bool').columns:
    independent_variables_encoded[col] = independent_variables_encoded[col].astype(int)

# Convert the outcome variable to numerical (e.g., 0 and 1)
# Assuming 'Active' is the positive class (1) and others are negative (0)
outcome_encoded = (outcome == 'Active').astype(int)

X = sm.add_constant(independent_variables_encoded)
model = sm.Logit(outcome_encoded, X)
result = model.fit()
print(result.summary())





Optimization terminated successfully.
         Current function value: 0.449470
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:            last_status   No. Observations:                 5255
Model:                          Logit   Df Residuals:                     5248
Method:                           MLE   Df Model:                            6
Date:                Sat, 15 Nov 2025   Pseudo R-squ.:                  0.2167
Time:                        04:08:46   Log-Likelihood:                -2362.0
converged:                       True   LL-Null:                       -3015.3
Covariance Type:            nonrobust   LLR p-value:                3.989e-279
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.9737      0.295      3.302      0.001       0.396       1.552